# **Install Dependencies**

In [ ]:
!pip uninstall -y sentence-transformers peft transformers
print("✅ Removed conflicting packages")

In [ ]:
!pip install "transformers>=4.46.0,<5.0.0" "diffusers>=0.26.0" accelerate sentencepiece
!pip install -q einops timm pillow fastapi uvicorn pyngrok nest-asyncio python-multipart
!pip install -q scikit-learn

print("✅ All packages installed")

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import torch
import io
import numpy as np
import base64
import json
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
from numpy.linalg import norm

from transformers import AutoModel
from diffusers import AutoPipelineForText2Image

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Environment stabilized on: {device}")

import nest_asyncio
import uvicorn
from fastapi import FastAPI, Form, HTTPException
from pyngrok import ngrok
from io import BytesIO
import threading
import random
import asyncio
import gc
from PIL import Image
from kaggle_secrets import UserSecretsClient

# **Load Model from Dataset**

embedding model (jina clip v2)

In [ ]:
# Load similarity model
MODEL_PATH = "jinaai/jina-clip-v2"
similarity_model = AutoModel.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
).to(device)
similarity_model.eval()
torch.cuda.empty_cache()
print("✅ Similarity model loaded!")

text to image model (SDXL Turbo)

In [ ]:
# Load generator model
GEN_MODEL_PATH = "stabilityai/sdxl-turbo"
gen_pipe = AutoPipelineForText2Image.from_pretrained(
    GEN_MODEL_PATH, 
    torch_dtype=torch.float16, 
    variant="fp16"
).to(device)
gen_pipe.enable_attention_slicing() 
print("✅ Generator loaded!")

# **Functions**

In [ ]:
def generate_image(prompt, seed=42):
    """Generate image with controllable seed"""
    
    generator = torch.Generator(device=device).manual_seed(seed)
    
    image = gen_pipe(
        prompt=prompt, 
        num_inference_steps=3,
        guidance_scale=0.0,
        generator=generator
    ).images[0]
    return image

def get_embedding(image_pil):
    """Convert PIL image to embedding vector"""
    with torch.no_grad():
        embedding = similarity_model.encode_image([image_pil])
    
    if isinstance(embedding, np.ndarray):
        return embedding
    else:
        return embedding.cpu().numpy()

def calculate_similarity(pre_embedded_vec, new_image_pil):
    """Compare pre-computed vector with new image"""
    new_vec = get_embedding(new_image_pil)
    original_vec = np.array(pre_embedded_vec).reshape(1, -1)
    score = cosine_similarity(original_vec, new_vec)[0][0]
    display_score = max(0, (score - 0.4) / 0.6) * 100
    return round(float(display_score), 2)

print("✅ Helper functions defined!")

Load Target Images from Dataset

In [ ]:
DATASET_BASE = "/kaggle/input/datasets/nirhazan/images-game"
PREGAME_DIR = f"{DATASET_BASE}/game_data/images"
EMBEDDINGS_FILE = f"{DATASET_BASE}/game_data/embeddings.json"

def load_game_embeddings():
    """Load embeddings with full case-insensitive support"""
    
    if not os.path.exists(EMBEDDINGS_FILE):
        raise FileNotFoundError(f"embeddings.json not found at: {EMBEDDINGS_FILE}")
    
    with open(EMBEDDINGS_FILE, 'r') as f:
        embeddings = json.load(f)
    
    # Get all actual filenames in the directory
    actual_files = os.listdir(PREGAME_DIR)
    filename_map = {f.lower(): f for f in actual_files}
    
    print(f"📁 Found {len(actual_files)} image files")
    
    # Fix paths with case-insensitive matching
    fixed_embeddings = {}
    
    for img_id, data in embeddings.items():
        expected_filename = os.path.basename(data['path'])
        actual_filename = filename_map.get(expected_filename.lower())
        
        if actual_filename:
            data['path'] = os.path.join(PREGAME_DIR, actual_filename)
            
            # Store with LOWERCASE key for case-insensitive lookup
            fixed_embeddings[img_id.lower()] = data
            print(f"✅ {img_id.lower()} → {actual_filename}")
        else:
            print(f"❌ {img_id}: File not found")
    
    print(f"\n✅ Loaded {len(fixed_embeddings)} target images")
    
    return fixed_embeddings

# Load embeddings
game_embeddings = load_game_embeddings()
print(f"\n🎮 Available: {list(game_embeddings.keys())}")

# **FastAPI**

In [ ]:
user_secrets = UserSecretsClient()
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
STATIC_DOMAIN = user_secrets.get_secret("NGROK_DOMAIN")
ngrok.set_auth_token(NGROK_TOKEN)

app = FastAPI()
nest_asyncio.apply()

gpu_lock = asyncio.Lock()

try:
    game_embeddings = load_game_embeddings()
except FileNotFoundError:
    print("❌ Cannot start - no target images!")
    raise

@app.get("/get_image")
async def get_image():
    try:
        image_id = random.choice(list(game_embeddings.keys()))
        img_path = game_embeddings[image_id]["path"]
        img = Image.open(img_path).convert("RGB")
        
        buffered = BytesIO()
        img.save(buffered, format="PNG")
        img_b64 = base64.b64encode(buffered.getvalue()).decode()
        
        return {"success": True, "image_b64": img_b64, "image_id": image_id}
    except Exception as e:
        return {"success": False, "error": str(e)}

@app.post("/play_round")
async def play_round(user_prompt: str = Form(...), target_image_id: str = Form(...)):
    normalized_id = target_image_id.lower()

    if normalized_id not in game_embeddings:
        return {"success": False, "error": f"Invalid target_image_id: {target_image_id}"}

    target_embedding = game_embeddings[normalized_id]["embedding"]

    async with gpu_lock:
        try:
            generated_img = generate_image(user_prompt, seed=42)
            final_score = calculate_similarity(target_embedding, generated_img)

            buffered = BytesIO()
            generated_img.save(buffered, format="PNG")
            img_b64 = base64.b64encode(buffered.getvalue()).decode()

            return {"success": True, "score": final_score, "image_b64": img_b64}

        except Exception as e:
            return {"success": False, "error": f"Generation failed: {str(e)}"}

        finally:
            if device == "cuda":
                torch.cuda.empty_cache()
            gc.collect()


@app.get("/health")
async def health():
    return {"status": "healthy", "queue_locked": gpu_lock.locked()}

# Start server
public_url = ngrok.connect(8000, bind_tls=True, hostname=STATIC_DOMAIN)
print(f"\n🚀 API LIVE at: {public_url}")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("✅ Server running in background thread!")

==================================================================

# **Test**

In [ ]:
import time
import requests
from IPython.display import Image as IPythonImage, display

# Your public ngrok URL
BASE_URL = "https://" + user_secrets.get_secret("NGROK_DOMAIN")

SLEEP_SECONDS = 1.0

def snooze(label=""):
    if label:
        print(f"⏳ Sleeping {SLEEP_SECONDS}s {label}...")
    time.sleep(SLEEP_SECONDS)

print("🧪 Testing API Endpoints...")
print("=" * 70)

target_id = None  # so Test 3 won't explode if Test 2 fails

# Test 1: Health Check
print("\n1️⃣ Testing /health endpoint...")
try:
    response = requests.get(f"{BASE_URL}/health")
    if response.status_code == 200:
        data = response.json()
        print(f"✅ Health check passed!")
        print(f"   Status: {data['status']}")
    else:
        print(f"❌ Failed with status {response.status_code}")
except Exception as e:
    print(f"❌ Error: {e}")

snooze("(after /health)")

# Test 2: Get Random Image
print("\n2️⃣ Testing /get_image endpoint...")
try:
    response = requests.get(f"{BASE_URL}/get_image")
    if response.status_code == 200:
        data = response.json()
        if data['success']:
            print(f"✅ Got target image!")
            print(f"   Image ID: {data['image_id']}")

            # Display the image
            import base64
            img_data = base64.b64decode(data['image_b64'])
            display(IPythonImage(img_data))

            # Save image_id for next test
            target_id = data['image_id']
        else:
            print(f"❌ API returned error: {data.get('error')}")
    else:
        print(f"❌ Failed with status {response.status_code}")
except Exception as e:
    print(f"❌ Error: {e}")

snooze("(after /get_image)")

# Test 3: Play Round
print("\n3️⃣ Testing /play_round endpoint...")
if not target_id:
    print("⚠️ Skipping Test 3 because target_id is missing (Test 2 failed).")
else:
    try:
        test_data = {
            "user_prompt": "a beautiful sunset over mountains",
            "target_image_id": target_id
        }

        response = requests.post(f"{BASE_URL}/play_round", data=test_data)

        if response.status_code == 200:
            data = response.json()
            if data['success']:
                print(f"✅ Round played successfully!")
                print(f"   Score: {data['score']}%")
                print(f"   Prompt: {test_data['user_prompt']}")

                # Display generated image
                import base64
                img_data = base64.b64decode(data['image_b64'])
                print("\n   Generated image:")
                display(IPythonImage(img_data))
            else:
                print(f"❌ API returned error: {data.get('error')}")
        else:
            print(f"❌ Failed with status {response.status_code}")
    except Exception as e:
        print(f"❌ Error: {e}")

snooze("(after /play_round)")

# Test 4: Case-insensitive ID test
print("\n4️⃣ Testing case-insensitive image IDs...")
try:
    test_ids = ["GAME_000", "Game_000", "game_000"]
    for test_id in test_ids:
        response = requests.post(
            f"{BASE_URL}/play_round",
            data={
                "user_prompt": "test prompt",
                "target_image_id": test_id
            }
        )
        if response.status_code == 200 and response.json().get('success'):
            print(f"   ✅ {test_id} works!")
        else:
            print(f"   ❌ {test_id} failed")

        snooze(f"(after {test_id})")
except Exception as e:
    print(f"❌ Error: {e}")

print("\n" + "=" * 70)
print("🎉 Testing complete!")